# gr00t n 1.5 deployment details
### they use GR1 as example, so we do not know much details about Droid.
- image size(1,256,256,3) | uint8| padding and resize, like pi0
- For droid, ext1 is left, ext2 is right
- they have already binarized gripper action in their code? action=1 means close, action=0 means open
- rotation is 'rpy' euler angle
- need to normalize the gripper state





### install pyzed
https://www.stereolabs.com/docs/development/python/install

(1) install ZED SDK

(2)
```cd "/usr/local/zed/" python3 get_python_api.py```

- In MPK, use ```conda activate gr00t``` or select env upper right

## select location


In [ ]:
import os
import torch
import gr00t

from gr00t.data.dataset import LeRobotSingleDataset
from gr00t.model.policy import Gr00tPolicy
import numpy as np

# set config

In [ ]:
# change the following paths
MODEL_PATH = "nvidia/GR00T-N1.5-3B"

# REPO_PATH is the path of the pip install gr00t repo and one level up
REPO_PATH = os.path.dirname(os.path.dirname(gr00t.__file__))
DATASET_PATH = os.path.join(REPO_PATH, "demo_data/robot_sim.PickNPlace")
EMBODIMENT_TAG = "oxe_droid"

device = "cuda" if torch.cuda.is_available() else "cpu"

# load model

In [ ]:
from gr00t.experiment.data_config import DATA_CONFIG_MAP

# can add [optional] denoising step in policy

data_config = DATA_CONFIG_MAP["oxe_droid"]
modality_config = data_config.modality_config()
modality_transform = data_config.transform()

policy = Gr00tPolicy(
    model_path=MODEL_PATH,
    embodiment_tag=EMBODIMENT_TAG,
    modality_config=modality_config,
    modality_transform=modality_transform,
    device=device,
)

# print out the policy model architecture
print(policy.model)

In [ ]:
modality_config = policy.modality_config

print(modality_config.keys())

for key, value in modality_config.items():
    if isinstance(value, np.ndarray):
        print(key, value.shape)
    else:
        print(key, value)


# load image preprocess functions

In [ ]:
from PIL import Image
def resize_with_pad(images: np.ndarray, height: int, width: int, method=Image.BILINEAR) -> np.ndarray:
    """Replicates tf.image.resize_with_pad for multiple images using PIL. Resizes a batch of images to a target height.

    Args:
        images: A batch of images in [..., height, width, channel] format.
        height: The target height of the image.
        width: The target width of the image.
        method: The interpolation method to use. Default is bilinear.

    Returns:
        The resized images in [..., height, width, channel].
    """
    # If the images are already the correct size, return them as is.
    if images.shape[-3:-1] == (height, width):
        return images

    original_shape = images.shape

    images = images.reshape(-1, *original_shape[-3:])
    resized = np.stack([_resize_with_pad_pil(Image.fromarray(im), height, width, method=method) for im in images])
    return resized.reshape(*original_shape[:-3], *resized.shape[-3:])


def _resize_with_pad_pil(image: Image.Image, height: int, width: int, method: int) -> Image.Image:
    """Replicates tf.image.resize_with_pad for one image using PIL. Resizes an image to a target height and
    width without distortion by padding with zeros.

    Unlike the jax version, note that PIL uses [width, height, channel] ordering instead of [batch, h, w, c].
    """
    cur_width, cur_height = image.size
    if cur_width == width and cur_height == height:
        return image  # No need to resize if the image is already the correct size.

    ratio = max(cur_width / width, cur_height / height)
    resized_height = int(cur_height / ratio)
    resized_width = int(cur_width / ratio)
    resized_image = image.resize((resized_width, resized_height), resample=method)

    zero_image = Image.new(resized_image.mode, (width, height), 0)
    pad_height = max(0, int((height - resized_height) / 2))
    pad_width = max(0, int((width - resized_width) / 2))
    zero_image.paste(resized_image, (pad_width, pad_height))
    assert zero_image.size == (width, height)
    return zero_image

# sanity check (only for debug)

In [ ]:
import numpy as np

step_data = {
    'video.exterior_image_1': np.random.randint(0, 255, (1, 256, 256, 3), dtype=np.uint8),
    'video.exterior_image_2': np.random.randint(0, 255, (1, 256, 256, 3), dtype=np.uint8),
    'video.wrist_image': np.random.randint(0, 255, (1, 256, 256, 3), dtype=np.uint8),
    'state.eef_position': np.random.randn(1, 3).astype(np.float32),
    'state.eef_rotation': np.random.randn(1, 3).astype(np.float32),
    'state.gripper_position': np.random.randn(1, 1).astype(np.float32),
    'annotation.language.language_instruction': ['pick up the red block and place it on the green platform']
}


In [ ]:
step_data['state.eef_position'].shape
step_data['state.eef_rotation'].shape
step_data['state.gripper_position'].shape

In [ ]:
predicted_action = policy.get_action(step_data)
for key, value in predicted_action.items():
    print(key, value.shape)

# GR00T inference client
- note: deoxys OSC controller uses Axis-Angle; action from gr00t is Euler angles

### for real world inference
- TODO: transfer the action preprocessing code(Euler angles to Axis-Angle) to ```VLA_deploy.ipynb```

### for WM inference
- pip install Pyro5
- just copy the URI from WM server every times

In [ ]:
# vla_policy_client
import Pyro5.api
import numpy as np
import time
from gr00t.data.transform.state_action import RotationTransform



# ns = Pyro5.api.locate_ns(host="a100-st-p4de24xlarge-236", port=9095)  # Locate the name server
# uri = ns.lookup("gr00t_WM_controller")  # Look up the registered object by name
# controller = Pyro5.api.Proxy(uri)

uri = "PYRO:obj_25abfc6cbd214d2bb50541a2a6eb21df@a100-st-p4de24xlarge-132:42975"  # copy the URI here
controller = Pyro5.api.Proxy(uri)

MAX_INFERENCE = 25

video_buffer = []

# prompt
prompt = ["pick up the red cube and place it to the back"]  # <-- change prompt

# dummy action
action = np.zeros((16,7), dtype=np.float32)
action_list = action.tolist()  # convert to list for sending

robot_pos = np.array([4.83606286e-01,  1.13770277e-01,  2.38324794e-01]).reshape(1, 3).astype(np.float32)
robot_rot = np.array([3.12726760e+00,  6.76223114e-02, -5.53913005e-02]).reshape(1, 3).astype(np.float32)
gripper_state = np.array([1.0]).reshape(1, 1).astype(np.float32)  # Assuming gripper state is a single value


for step in range(MAX_INFERENCE):

    # print(f"\n=== Step {step} ===")
    data_to_send = {
        "action": action_list,
        "step": step
    }
    obs = controller.step(data_to_send)  # result is dict
    img_left = obs["left_image"]
    img_left = np.array(img_left).transpose(1, 2, 0).astype(np.uint8)   # Convert PIL image to numpy array
    print(f"Step {step}, Image shape: {img_left.shape}")
    img_left = resize_with_pad(img_left, 256, 256)  # Resize image to 256x256
    img_left = img_left[None, ...]     # shape: (1, 256, 256, 3)





    # step_data = {
    #     'video.exterior_image_1': img_left,
    #     'video.exterior_image_2': img_right,
    #     'video.wrist_image': img_wrist,
    #     'state.eef_position': np.array(obs['robot_pos'], dtype=np.float32),  # (1, 3)
    #     'state.eef_rotation': np.array(obs['robot_rot'], dtype=np.float32),  # (1, 3)
    #     'state.gripper_position': np.array(obs['gripper_state'], dtype=np.float32),  # (1, 1)
    #     'annotation.language.language_instruction': prompt
    # }
    step_data = {
        'video.exterior_image_1': img_left,
        'video.exterior_image_2': np.zeros_like(img_left),
        'video.wrist_image': np.zeros_like(img_left), 
        'state.eef_position': robot_pos,
        'state.eef_rotation': robot_rot,
        'state.gripper_position': gripper_state,
        'annotation.language.language_instruction': prompt
    }



    predicted_action = policy.get_action(step_data)
    scaling_factor = 0.01    # a magic number
    pos = predicted_action['action.eef_position_delta'] * scaling_factor  # (16, 3)
    rot = predicted_action['action.eef_rotation_delta'] * scaling_factor  # (16, 3)
    grip = predicted_action['action.gripper_position']  # (16,)
    if grip.ndim == 1:
        grip = grip[:, np.newaxis]

    action_concat = np.concatenate([pos, rot, grip], axis=-1)
    action_list = action_concat.tolist()  # convert to list for sending
    print(action_list)

    # update robot state
    robot_pos += pos[:5].sum(axis=0, keepdims=True)
    robot_rot += rot[:5].sum(axis=0, keepdims=True)
    gripper_state = grip[4].reshape(1, 1) 
    print(f"Robot position: {robot_pos.shape}, Robot rotation: {robot_rot.shape}, Gripper state: {gripper_state.shape}")


In [ ]:
from gr00t.data.transform.state_action import RotationTransform

euler_to_axis_angle_tform = RotationTransform(
    from_rep="euler_angles_rpy",
    to_rep="axis_angle",
)

print(euler_to_axis_angle_tform.forward(torch.from_numpy(rot)))

# save gif

In [ ]:
import imageio
import cv2


gif_path = "gr00t_deploy_2.gif"
imageio.mimsave(gif_path, video_buffer, duration=0.5)
print(f"GIF saved:{gif_path}")

In [ ]:
from IPython.display import Image as IPyImage
IPyImage(filename="gr00t_deploy_2.gif")
